# 🫀 Coronary Vessel Segmentation — Colab Runner

**AI512 Project** · Hybrid CNN–Mamba U-Net on DCA1  
Run this notebook on a Colab GPU runtime (T4 or better).

---
## ⚡ Workflow
1. Mount Drive → clone/pull repo → install deps
2. Upload DCA1 data to Drive once
3. Run training cells
4. Checkpoints auto-save to Drive

In [ ]:
# ============================================================
# CELL 1: Check GPU
# ============================================================
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ============================================================
# CELL 2: Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# Set your Drive path — change if needed
DRIVE_PROJECT = '/content/drive/MyDrive/coronary-vessel-segmentation'
DRIVE_DATA    = '/content/drive/MyDrive/DCA1_data'  # where you upload DCA1 images/masks
print('Drive mounted.')

In [ ]:
# ============================================================
# CELL 3: Clone / pull repo
# ============================================================
import os

REPO_URL = 'https://github.com/shlok923/coronary-vessel-segmentation.git'
PROJECT_DIR = '/content/coronary-vessel-segmentation'

if os.path.exists(PROJECT_DIR):
    print('Repo already cloned — pulling latest...')
    !cd {PROJECT_DIR} && git pull
else:
    print('Cloning repo...')
    !git clone {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ============================================================
# CELL 4: Install dependencies
# ============================================================
!pip install -q -r requirements.txt

# Optional: Mamba SSM (comment out if not needed or fails)
# !pip install -q causal-conv1d==1.2.0
# !pip install -q mamba-ssm==1.2.0

print('Dependencies installed.')

In [ ]:
# ============================================================
# CELL 5: Link DCA1 data from Drive
# ============================================================
# Upload your DCA1 images and masks to Google Drive at DRIVE_DATA/images/ and DRIVE_DATA/masks/
# Then run this cell to symlink them into the project.

import os

data_dir = os.path.join(PROJECT_DIR, 'data', 'DCA1')
os.makedirs(data_dir, exist_ok=True)

for subdir in ['images', 'masks', 'skeletons']:
    src = os.path.join(DRIVE_DATA, subdir)
    dst = os.path.join(data_dir, subdir)
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)
        print(f'Linked: {dst} → {src}')
    elif os.path.exists(dst):
        print(f'Already exists: {dst}')
    else:
        print(f'WARNING: Source not found: {src} — upload DCA1 data to Drive first!')

# Check
n_images = len(os.listdir(os.path.join(data_dir, 'images'))) if os.path.exists(os.path.join(data_dir, 'images')) else 0
print(f'Images found: {n_images}')

In [ ]:
# ============================================================
# CELL 6: Data sanity check
# ============================================================
!python -m src.sanity_check

In [ ]:
# ============================================================
# CELL 7: Run metric unit tests
# ============================================================
!pip install -q pytest
!pytest tests/test_metrics.py -v

In [ ]:
# ============================================================
# CELL 8: Train BASELINE U-Net (M1)
# ============================================================
!python -m src.train --config configs/baseline.yaml

In [ ]:
# ============================================================
# CELL 9: Train Hybrid U-Net (M2, no innovation)
# ============================================================
!python -m src.train --config configs/hybrid.yaml

In [ ]:
# ============================================================
# CELL 10: Train Hybrid U-Net + Innovation I2 (PROPOSED)
# ============================================================
!python -m src.train --config configs/hybrid_i2.yaml

In [ ]:
# ============================================================
# CELL 11: Final evaluation on TEST SET
# ⚠️  Only run this ONCE — after all training is complete!
# ============================================================
!python -m src.evaluate --all --split test

In [ ]:
# ============================================================
# CELL 12: Profile models (params, FLOPs, speed)
# ============================================================
!python -m src.profile --config configs/baseline.yaml
!python -m src.profile --config configs/hybrid_i2.yaml

In [ ]:
# ============================================================
# CELL 13: Copy results to Google Drive
# ============================================================
import shutil, os

results_src = os.path.join(PROJECT_DIR, 'results')
results_dst = os.path.join(DRIVE_PROJECT, 'results')
os.makedirs(results_dst, exist_ok=True)

shutil.copytree(results_src, results_dst, dirs_exist_ok=True)
print(f'Results copied to Drive: {results_dst}')

---
## 📋 Notes
- **Test set is never used during training** — only in Cell 11
- Checkpoints saved to `results/checkpoints/{experiment}/best.pth`
- TensorBoard logs in `results/runs/{experiment}/` — open with `%tensorboard --logdir results/runs`
- If Colab session disconnects mid-training, re-run from Cell 3 (git pull) and then the training cell